# ClaimClerk extraction chain

Builds and registers the ClaimClerk extraction chain. Demonstrates blueprint
objectives 1.3, 3.1, 4.1, 4.3, 4.5 end-to-end:

1. **1.3 / 4.3** Three-component chain (prompt → LLM → JSON output)
2. **3.1** LangChain `ChatDatabricks` as the LLM client
3. **4.1** PyFunc subclass with pre/post-processing (PII strip +
   JSON schema validation + envelope wrap)
4. **4.5** Register the model to Unity Catalog via
   `mlflow.pyfunc.log_model` with `infer_signature` + aliases

The chain wraps the `claimclerk_extraction@champion` prompt (registered by c0301).
The artifact lands at
`genaicert.pawshield.claimclerk_extraction_chain` with `@champion`
alias on v1.

Companion module: `claimclerk_extraction_chain.py` (the standalone
.py that `mlflow.pyfunc.log_model(python_model=...)` re-executes
fresh at serve time).

Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent

## 0. Setup

In [0]:
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install --quiet --force-reinstall "mlflow[databricks]>=3.12,<3.13" "databricks-vectorsearch<0.74" databricks-sdk databricks-langchain jsonschema

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json
import os
import sys

import mlflow
import pandas as pd
from mlflow.models import infer_signature, ModelConfig

# Make the sibling module importable. Databricks workspace notebooks live
# alongside .py files in the same directory; this prepends that directory
# to sys.path so `import claimclerk_extraction_chain` works in-notebook
# BEFORE log_model packages the same file as a deployed artifact.
notebook_dir = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
if notebook_dir and notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")

CATALOG = dbutils.widgets.get("catalog")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
SCHEMA = "pawshield"

PROMPT_NAME = f"{CATALOG}.{SCHEMA}.claimclerk_extraction"
PROMPT_URI = f"prompts:/{PROMPT_NAME}@champion"

CHAIN_FILE = "claimclerk_extraction_chain.py"
BARE_CHAIN_FILE = "claimclerk_extraction_bare.py"
REGISTERED_MODEL = f"{CATALOG}.{SCHEMA}.claimclerk_extraction_chain"
BARE_REGISTERED_MODEL = f"{CATALOG}.{SCHEMA}.claimclerk_extraction_bare"

print(f"Catalog:             {CATALOG}")
print(f"LLM endpoint:        {LLM_ENDPOINT}")
print(f"Prompt URI:          {PROMPT_URI}")
print(f"Chain artifact:      {CHAIN_FILE}")
print(f"Registered model:    {REGISTERED_MODEL}")
print(f"Bare-flavor model:   {BARE_REGISTERED_MODEL}")

Catalog:             genaicert
LLM endpoint:        databricks-meta-llama-3-1-8b-instruct
Prompt URI:          prompts:/genaicert.pawshield.claimclerk_extraction@champion
Chain artifact:      claimclerk_extraction_chain.py
Registered model:    genaicert.pawshield.claimclerk_extraction_chain
Bare-flavor model:   genaicert.pawshield.claimclerk_extraction_bare


## 1. Resolve the prompt from the Registry (build-time pattern)

The chain calls `mlflow.genai.load_prompt(...)` at BUILD time, not
inside `load_context`. The resolved template bakes into the logged
artifact, so the served version is byte-exact replayable from the
registered model alone — no live Registry lookup at serve time (the
audit-replay property).

The runtime alternative — resolve in `load_context` so an alias-move
alone updates production without re-logging — is the shape to prefer
for fast prompt promotion. The PolicyPal chain (policypal_chain.py) carries that branch
as a fallback but, as shipped, bakes the template at log_model, so its
promotion is re-log + redeploy. The trade-off: fast promotion vs
byte-exact audit replay.

In [0]:
from claimclerk_extraction_chain import DEFAULT_PROMPT_TEMPLATE

try:
    resolved_template = mlflow.genai.load_prompt(PROMPT_URI).template
    prompt_source = PROMPT_URI
except Exception as e:
    print(f"  Registry lookup failed ({e.__class__.__name__}); "
          f"falling back to inline template — run c0301 first to "
          f"register {PROMPT_NAME}@champion.")
    resolved_template = DEFAULT_PROMPT_TEMPLATE
    prompt_source = "inline-default"

print(f"Prompt resolved from: {prompt_source}")
print(f"Template length:      {len(resolved_template)} chars")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c98b88e8-1f20-4ed2-a143-3861ace2b8f8/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Prompt resolved from: prompts:/genaicert.pawshield.claimclerk_extraction@champion
Template length:      1711 chars


## 2. In-process smoke test

Build the PyFunc with the resolved template + LLM endpoint, run
one Sarah-style email through it, confirm the JSON validates.
Catches schema drift, PII redaction misses, and prompt regressions
BEFORE the log_model packages the artifact.

In [0]:
from claimclerk_extraction_chain import ClaimClerkExtraction

model_config = {
    "llm_endpoint":    LLM_ENDPOINT,
    "prompt_template": resolved_template,
    "prompt_uri":      prompt_source,
    "version_marker":  "v1-launch",
}

# Synthesize a Databricks-style context object for in-process testing.
class _LocalContext:
    def __init__(self, cfg): self.model_config = cfg

local_chain = ClaimClerkExtraction()
local_chain.load_context(_LocalContext(model_config))

sample_email = (
    "From: sarah.chen@gmail.com\n"
    "Subject: Appeal — claim CLM-2026-03-00471\n\n"
    "Hi PawShield team — I'm appealing the denial on Biscuit's chronic "
    "kidney disease claim. I believe Section 4.7 of my Texas PawPlus policy "
    "(POL-2025-01-CHENB02) covers the second visit. Reach me at "
    "512-555-0188.\n\n"
    "Sarah Chen (CUST-CHEN-001)"
)
smoke_input = pd.DataFrame([{"email_body": sample_email}])

smoke_out = local_chain.predict(None, smoke_input)
print(json.dumps(smoke_out[0], indent=2))

{
  "extracted": "{\"claim_id\": \"CLM-2026-03-00471\", \"customer_id\": \"CUST-CHEN-001\", \"pet_name\": \"Biscuit\", \"contact_phone\": null, \"intent\": \"appeal\", \"sentiment\": \"neutral\", \"urgency\": \"low\"}",
  "version_marker": "v1-launch",
  "prompt_uri": "prompts:/genaicert.pawshield.claimclerk_extraction@champion",
  "request_timestamp_unix": 1780765019.9688067
}


## 3. Set the MLflow registry to Unity Catalog

MLflow 3.x defaults to UC; on MLflow 2.x the call is mandatory.
Without it, three-part names are rejected and the model silently
lands in the workspace-scoped Model Registry instead — the missing one-liner.

In [0]:
mlflow.set_registry_uri("databricks-uc")
print(f"Registry URI: {mlflow.get_registry_uri()}")

Registry URI: databricks-uc


## 4. Infer the signature

UC rejects any `register_model` call without input + output
signatures. The cleanest authoring path is
`signature=infer_signature(input_example, output_example)` passed
directly on `log_model`. (`mlflow.models.set_signature` is the
recovery API for already-logged-without-signature models.)

In [0]:
input_example = pd.DataFrame([{"email_body": sample_email}])
output_example = smoke_out
signature = infer_signature(input_example, output_example)
print(signature)

inputs: 
  ['email_body': string (required)]
outputs: 
  ['extracted': string (required), 'version_marker': string (required), 'prompt_uri': string (required), 'request_timestamp_unix': double (required)]
params: 
  None



## 5. Log via `mlflow.pyfunc.log_model` (code-based logging)

`python_model=<path-to-.py>` is code-based logging:
`claimclerk_extraction_chain.py` is the artifact MLflow re-executes
fresh at serve time, not a pickled object. The `mlflow.models.set_model(...)`
line at the bottom of the chain module is what tells MLflow which
class to re-instantiate.

`extra_pip_requirements` (not `pip_requirements`) layers extra
pins on top of auto-inference. Using `pip_requirements` instead
would replace the auto-inferred list and drop transitively-needed
packages — the most common ModuleNotFoundError at serve time.

In [0]:
with mlflow.start_run(run_name="claimclerk_extraction_chain_v1"):
    logged = mlflow.pyfunc.log_model(
        name="claimclerk_extraction_chain",
        python_model=CHAIN_FILE,
        model_config=model_config,
        signature=signature,
        input_example=input_example,
        registered_model_name=REGISTERED_MODEL,
        extra_pip_requirements=[
            "databricks-langchain",
            "langchain-core>=0.3",
            "jsonschema",
        ],
    )

print(f"Logged: {logged.model_uri}")
print(f"Registered: {REGISTERED_MODEL} v{logged.registered_model_version}")

🔗 View Logged Model at: https://<workspace>.cloud.databricks.com/ml/experiments/4306497714730821/models/m-2e116528228e4913a57b278f0ae3b7c2?o=<WORKSPACE_ID>
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c98b88e8-1f20-4ed2-a143-3861ace2b8f8/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/06/06 16:57:04 INFO mlflow.pyfunc: Validating input example against model signature
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c98b88e8-1f20-4ed2-a143-3861ace2b8f8/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_mode

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '6' of model 'genaicert.pawshield.claimclerk_extraction_chain': https://<workspace>.cloud.databricks.com/explore/data/models/genaicert/pawshield/claimclerk_extraction_chain/version/6?o=<WORKSPACE_ID>


Logged: models:/m-2e116528228e4913a57b278f0ae3b7c2
Registered: genaicert.pawshield.claimclerk_extraction_chain v6


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias="champion",
    version=logged.registered_model_version,
)
print(f"@champion → v{logged.registered_model_version}")

@champion → v6


## 6. Verify by loading from UC and invoking

Round-trips the registered model through `pyfunc.load_model`,
re-runs the smoke-test email through it, and confirms the
envelope shape matches the in-process run. Catches code-based-
logging mistakes (missing `set_model`, sys.path issues, module-
scope import of un-installed packages) BEFORE a downstream promotion
pipeline picks the chain up.

In [0]:
loaded = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@champion")
loaded_out = loaded.predict(input_example)
print(json.dumps(loaded_out[0], indent=2))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c98b88e8-1f20-4ed2-a143-3861ace2b8f8/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


{
  "extracted": "{\"claim_id\": \"CLM-2026-03-00471\", \"customer_id\": \"CUST-CHEN-001\", \"pet_name\": \"Biscuit\", \"contact_phone\": null, \"intent\": \"appeal\", \"sentiment\": \"neutral\", \"urgency\": \"low\"}",
  "version_marker": "v1-launch",
  "prompt_uri": "prompts:/genaicert.pawshield.claimclerk_extraction@champion",
  "request_timestamp_unix": 1780765041.7716103
}


## 7. Alternate flavor — `mlflow.langchain.log_model` (code-based)

`mlflow.langchain.log_model` is the alternate flavor for pure
LangChain Runnables with no PyFunc pre/post-processing. With
**LangChain v1+** the API requires **code-based-logging**:
`lc_model=` must be a *path* to a .py file that constructs the
chain and registers it via `mlflow.models.set_model(chain)`.
The live-chain-object form (`lc_model=<runnable>`) is deprecated
and raises an `MlflowException` on v1+.

Pedagogical takeaway: the langchain vs pyfunc choice is no
longer about "code path vs file path" — both flavors now use the
same code-based-logging shape. The remaining difference is which
lineage MLflow captures (LangChain-specific chain spec for
langchain.log_model; generic PyFunc signature for pyfunc.log_model).

In [0]:
# Import the module to construct + smoke-test the chain locally before
# log_model packages the same file. Same `sys.path` trick used for the
# main chain module above.
#
# `invalidate_caches()` is required when a sibling .py is uploaded
# AFTER the Python kernel started — Python caches what's visible at
# each sys.path entry at first-touch, so a freshly uploaded workspace
# file is invisible to `import` until the cache is rebuilt. The
# subsequent `reload` picks up any later edits to the same file.
import importlib
importlib.invalidate_caches()
import claimclerk_extraction_bare
importlib.reload(claimclerk_extraction_bare)
from claimclerk_extraction_bare import bare_chain

bare_out = bare_chain.invoke({"email_body": sample_email})
print(bare_out[:300] + ("..." if len(bare_out) > 300 else ""))

{
  "claim_id": "CLM-2026-03-00471",
  "customer_id": "CUST-CHEN-001",
  "pet_name": "Biscuit",
  "contact_phone": "512-555-0188",
  "intent": "appeal",
  "sentiment": "frustrated",
  "urgency": "medium"
}


In [0]:
# Pass the FILE PATH to `lc_model=`, not the chain object. MLflow re-
# executes the file at serve time and uses the runnable registered via
# `mlflow.models.set_model(...)` inside that file.
with mlflow.start_run(run_name="claimclerk_extraction_bare_langchain"):
    bare_logged = mlflow.langchain.log_model(
        lc_model=BARE_CHAIN_FILE,
        name="claimclerk_extraction_bare",
        input_example={"email_body": sample_email},
        registered_model_name=BARE_REGISTERED_MODEL,
        extra_pip_requirements=["databricks-langchain", "langchain-core>=0.3"],
    )
print(f"Bare-flavor logged: {bare_logged.model_uri}")
print(f"Registered: {BARE_REGISTERED_MODEL} v{bare_logged.registered_model_version}")

🔗 View Logged Model at: https://<workspace>.cloud.databricks.com/ml/experiments/4306497714730821/models/m-ef3e760ad7c0445897aa4e0c2009977f?o=<WORKSPACE_ID>
2026/06/06 16:57:25 INFO mlflow: Attempting to auto-detect Databricks resource dependencies for the current langchain model. Dependency auto-detection is best-effort and may not capture all dependencies of your langchain model, resulting in authorization errors when serving or querying your model. We recommend that you explicitly pass `resources` to mlflow.langchain.log_model() to ensure authorization to dependent resources succeeds when the model is deployed.
2026/06/06 16:57:34 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-ef3e760ad7c0445897aa4e0c2009977f
2026/06/06 16:57:34 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
Registered model 'genaicert.pawshield.claimclerk_extraction_bare' already exists. Creating a new version of this 

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '5' of model 'genaicert.pawshield.claimclerk_extraction_bare': https://<workspace>.cloud.databricks.com/explore/data/models/genaicert/pawshield/claimclerk_extraction_bare/version/5?o=<WORKSPACE_ID>


Bare-flavor logged: models:/m-ef3e760ad7c0445897aa4e0c2009977f
Registered: genaicert.pawshield.claimclerk_extraction_bare v5


## 8. Summary

Two UC-registered artefacts now exist:

| Model | Flavor | Pre/post-processing | Use |
|---|---|---|---|
| `genaicert.pawshield.claimclerk_extraction_chain@champion` | `pyfunc` | PII strip + JSON validate + envelope | Production-shape; byte-exact audit-replay base |
| `genaicert.pawshield.claimclerk_extraction_bare` | `langchain` | none | Teaching demo of the langchain-flavor log path |

The ClaimClerk agent (c0901) builds fresh as a ResponsesAgent
with MCP tools — it doesn't consume either artifact. The
registration **pattern** (PyFunc subclass → infer_signature →
pyfunc.log_model → set_registered_model_alias) is what c0901 reuses, not the artefact itself.

End of c0701.